In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Load the model
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 851.16it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
df = pd.read_csv(r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\all\all_products_coarsetag_clean.csv')
categories = df.coarse_category.unique() # List of categories
indices = {}
texts = {}
embeddings = {}
for cat in categories: # Loop over every categories
    indices[cat] = df.index[df['coarse_category'] == cat].tolist()
    c_indices = indices[cat]
    c_text = []
    for idx in c_indices:
        c_text.append(df.loc[idx]["Canonical Text"])
    texts[cat] = c_text
for key in texts.keys():
    embeddings[key] = model.encode(texts[key], normalize_embeddings=True)
print(embeddings)

{'baby_kids': array([[ 0.01531457,  0.01174212, -0.00183317, ..., -0.00648097,
        -0.02933029,  0.02989174],
       [ 0.02777348, -0.0338442 ,  0.00123037, ..., -0.00979396,
         0.01183445,  0.04419443],
       [-0.04106564,  0.09564053, -0.05155187, ..., -0.05531142,
         0.00367741,  0.01646636],
       ...,
       [ 0.01727114,  0.08870225,  0.04599663, ..., -0.00939655,
         0.00686249,  0.08062412],
       [-0.05302559,  0.10134032,  0.02420073, ..., -0.041299  ,
         0.00226201,  0.04892866],
       [-0.10295274,  0.09213969, -0.01847445, ..., -0.02023174,
         0.05993472, -0.02628417]], shape=(60, 384), dtype=float32), 'bathroom': array([[ 0.06642017,  0.04481832,  0.07505485, ..., -0.03420387,
        -0.01693692,  0.03538502],
       [-0.04282946,  0.06766585,  0.02593086, ...,  0.03376086,
        -0.02130918,  0.02084591],
       [-0.05665835,  0.09318525, -0.01891551, ...,  0.06071994,
         0.0174883 ,  0.01811072],
       ...,
       [-0.02358

In [12]:
def get_top_recommendations(matrix, index=0, top_n=10):
    """
    Retrieves the top N similar items for a given index based on a similarity matrix.
    
    Args:
        matrix: The similarity matrix (Tensor or NumPy array).
        index (int): The row index to analyze.
        top_n (int): Number of top results to return (including the item itself).
        
    Returns:
        list: A list of tuples containing (index, score) for the top matches.
    """
    # Convert to numpy if it's a tensor
    if hasattr(matrix, 'numpy'):
        sim_matrix = matrix.numpy()
    else:
        sim_matrix = matrix

    # Get the scores for the specific row
    row_scores = sim_matrix[index]

    # Get indices of the highest scores in descending order
    # argsort sorts ascending, so we flip it with [::-1]
    sorted_indices = row_scores.argsort()[::-1]
    top_indices = sorted_indices[:top_n]
    top_scores = row_scores[top_indices]

    # Create list of (index, score) tuples, skipping the first item 
    # (assuming the first item is the query item itself with a self-similarity of 1.0)
    product_list = list(zip(top_indices[1:], top_scores[1:]))
    
    return product_list

from sentence_transformers import SentenceTransformer, util

thresholds = {
    "baby_kids": 0.62,
    "bedding": 0.72,
    "bathroom": 0.68,
    "dining": 0.74,
    "storage": 0.80,
    "clothing": 0.72,
    "loungeware": 0.72,
    "accessories": 0.80,
    "home_decor": 0.75,
    "kitchen": 0.75
}

all_edges = []

matrices = {}
for key in texts.keys():
    matrices[key] = util.cos_sim(embeddings[key], embeddings[key])
for cat in categories:
    recommendations = get_top_recommendations(matrices[cat], index=len(matrices[cat])-1, top_n=11)
    for i in range(len(texts[cat])):
        print(recommendations[i])
        if recommendations[i][1] >= thresholds[cat]:
            global_i = indices[cat][i]
            global_j = indices[cat][neighbor_local_idx]
            all_edges.append((global_i, global_j))
print(all_edges)

(np.int64(23), np.float32(0.8252724))


NameError: name 'neighbor_local_idx' is not defined